# 04C duration adjusted robustness

**Purpose.** Implement the authorized Phase 1 repair with preserved source values. Population: All 1,048,575 rows for matching/counterexamples; fixed regression sample 200,000, seed 2026.

**Hypothesis.** FX and/or gap can have conditional associations even if their observed median penalties are not universally positive.

**Methodology.** OLS for log1p(seconds) and binomial GLM for >900s include FX, gap, participants, route, currency group, log1p(USD) and party. HC1 pointwise intervals; five random/time partitions. Binary >300/>1800 sensitivity uses the same full regression sample. Exact shared strata require >=20/arm; compare adjacent FX within all other strata and gap within route/participants/FX/currency group/amount band/party. Min-arm overlap weights and independent-cell analytic intervals are reused from the verified helper. No unsupported stratum is extrapolated. Currency groups are the top 20 by population frequency plus OTHER; no outcome screening is used.

All duration calculations are seconds; rate differences are percentage points. Amount is USD-normalized. UNLABELED is not verified normal. Source placeholders, outliers and labels are retained. Private output tables can contain competition data and must not be committed or published.


In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Markdown, Image
ROOT = Path.cwd().resolve()
if not (ROOT / "data/cleaned/swift_payments_cleaned.csv").exists():
    ROOT = ROOT.parent
assert Path(sys.executable).resolve() == (ROOT / ".venv/Scripts/python.exe").resolve()
sys.path.insert(0, str(ROOT / "src"))
from phase1_analysis import Phase, adjusted_duration
phase = Phase(ROOT, '04C_duration_adjusted_robustness', 'duration/adjusted')


## Code and evidence

The implementation is in `src/phase1_analysis.py`. All displayed values and exports are calculated; group denominators and support restrictions are included in each output table. Models are associational. Fixed sampling avoids repeated fitting on the entire text-heavy dataset; full-population descriptive evidence remains intact.


In [ ]:
effects, matching, fx_counterexamples, gap_counterexamples = adjusted_duration(phase)
display(effects)
display(matching)

## Interpretation, limitations and business implications

Interpret only the estimated populations. Log-duration contrasts are not raw median changes; odds ratios are not risk ratios. Rate denominators are group-specific and slow thresholds are exploratory cut-offs, not Swift SLAs. Internal random and chronological checks are not external validation. Related payment families, residual bank/currency composition, coarse amount controls and sparse support limit causal interpretation. BIC outcome-overlap selection changes the estimand.

Use the evidence for source authority, supported segment monitoring and investigation priorities; no savings, fraud accuracy or operational improvement is inferred. An unexplained residual or insufficient common support is a valid result.


In [ ]:
report = phase.finish(["Full-sample adjusted effects: " + effects.loc[effects.split.eq("full")].to_json(orient="records"), "Supported full matched comparisons: " + matching.loc[matching.split.eq("full")].to_json(orient="records")], ["Pointwise conditional uncertainty; no causal or external-validation claim.", "Restricted regression/common-support populations must not be generalized to the full network."])
display(Markdown("## Final findings and limitations\n" + "\n\n".join(report["findings"])))
display(Markdown("Generated tables: `" + str(phase.out.relative_to(ROOT)) + "`; charts: `" + str(phase.charts.relative_to(ROOT)) + "`. Source hashes and validation checks are in the completion manifest."))
for chart in sorted(phase.charts.glob("*.png")):
    display(Image(filename=str(chart)))
